# From the Dengue Exercise to ARGO - Lane B (backup: the agent's output, captured)

**SISMID 2026 - Day 2, 1:30.** Continues Mauricio's dengue exercise 2: we grow a static
least-squares line into **ARGO**, step by step, scored out of sample on 2007-2011.

> **What this notebook is.** Each cell is a captured example of what a coding agent
> (Codex, Claude Code, or Antigravity CLI) produces from the prompts in the **Lane A**
> notebook. Run it top to bottom as a backup, or read it to see the target. Data:
> `MX_Dengue_trends.csv` (`Dengue CDC` = cases; four Google search terms).


## Step 0: load, and recap the static model (from Part 1)

Static = fit once on the first 36 months (2004-2006), predict 2007-2011 from `dengue`.


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os
from sklearn.linear_model import LinearRegression

CANDIDATES = ['../data/MX_Dengue_trends.csv', 'data/MX_Dengue_trends.csv', './MX_Dengue_trends.csv']
path = next((p for p in CANDIDATES if os.path.exists(p)), CANDIDATES[0])
df = pd.read_csv(path); df['Date'] = pd.to_datetime(df['Date'])
SEARCH = ['dengue', 'sintomas de dengue', 'mosquito', 'dengue sintomas']
TARGET = 'Dengue CDC'
df.head()


In [ ]:
# --- Produced by the agent from the Plan A / Step 0 prompt ---
df_train = df.iloc[:36]
m = LinearRegression().fit(df_train[['dengue']], df_train[TARGET])
df_valid = df.iloc[36:].copy()
pred_static = m.predict(df_valid[['dengue']])

def rmse(truth, pred):
    return float(np.sqrt(np.mean((np.asarray(truth) - np.asarray(pred))**2)))
print('static RMSE:', round(rmse(df_valid[TARGET], pred_static), 1))


## Step 1: dynamic (sliding-window) training

Refit on the previous 36 months at every step, then predict the next month. The model
stays current instead of frozen in 2006.


In [ ]:
# --- Produced by the agent from the Plan A / Step 1 prompt ---
def dynamic_predict(frame, features, window=36):
    preds = np.zeros(frame.shape[0] - window)
    for month in range(window, frame.shape[0]):
        tr = frame.iloc[max(month - window, 0):month]
        lr = LinearRegression().fit(tr[features], tr[TARGET])
        preds[month - window] = lr.predict(frame.iloc[month:month+1][features])[0]
    return preds

pred_dyn = dynamic_predict(df, ['dengue'])
print('dynamic (1 term) RMSE:', round(rmse(df_valid[TARGET], pred_dyn), 1))


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(df_valid['Date'], df_valid[TARGET], 'k', label='Cases')
plt.plot(df_valid['Date'], pred_static, 'r', label='Static')
plt.plot(df_valid['Date'], pred_dyn, 'g', label='Dynamic')
plt.legend(); plt.title('Static vs dynamic (one term)'); plt.tight_layout(); plt.show()


## Step 2: add all four search terms


In [ ]:
# --- Produced by the agent from the Plan A / Step 2 prompt ---
pred_dyn_all = dynamic_predict(df, SEARCH)
print('dynamic (all terms) RMSE:', round(rmse(df_valid[TARGET], pred_dyn_all), 1))


## Step 3: add autoregression = ARGO

Add the disease's own recent past (last two months of cases) as features. Dynamic
training + Google terms + autoregression is a basic **ARGO** model.


In [ ]:
# --- Produced by the agent from the Plan A / Step 3 prompt ---
dfa = df.copy()
dfa['ar1'] = dfa[TARGET].shift(1)
dfa['ar2'] = dfa[TARGET].shift(2)
ARGO_FEATURES = SEARCH + ['ar1', 'ar2']

def dynamic_predict_ar(frame, features, window=36):
    preds = np.zeros(frame.shape[0] - window)
    for month in range(window, frame.shape[0]):
        tr = frame.iloc[max(month - window, 2):month]   # first 2 rows have NaN AR terms
        lr = LinearRegression().fit(tr[features], tr[TARGET])
        preds[month - window] = lr.predict(frame.iloc[month:month+1][features])[0]
    return preds

pred_argo = dynamic_predict_ar(dfa, ARGO_FEATURES)
print('ARGO (dynamic + terms + AR2) RMSE:', round(rmse(df_valid[TARGET], pred_argo), 1))


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(df_valid['Date'], df_valid[TARGET], 'k', label='Cases')
plt.plot(df_valid['Date'], pred_static, 'r', label='Static')
plt.plot(df_valid['Date'], pred_dyn, 'g', label='Dynamic 1-term')
plt.plot(df_valid['Date'], pred_dyn_all, 'b', label='Dynamic all terms')
plt.plot(df_valid['Date'], pred_argo, 'purple', label='ARGO')
plt.legend(); plt.title('From static line to ARGO'); plt.tight_layout(); plt.show()


## Step 4: the error drops at every step


In [ ]:
# --- Produced by the agent from the Plan A / Step 4 prompt ---
for name, p in [('static (1 term)', pred_static), ('dynamic (1 term)', pred_dyn),
                ('dynamic (all terms)', pred_dyn_all), ('ARGO (+AR2)', pred_argo)]:
    print(f'{name:22s} RMSE = {rmse(df_valid[TARGET], p):8.1f}')


## Stretch: the full ARGO uses Lasso

With many correlated search terms, the real ARGO adds an L1 penalty so useless terms
drop out. Here it is with a handful of terms; the win grows with more terms.


In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def dynamic_predict_lasso(frame, features, window=36):
    preds = np.zeros(frame.shape[0] - window)
    for month in range(window, frame.shape[0]):
        tr = frame.iloc[max(month - window, 2):month]
        model = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=50000))
        model.fit(tr[features], tr[TARGET])
        preds[month - window] = model.predict(frame.iloc[month:month+1][features])[0]
    return preds

pred_lasso = dynamic_predict_lasso(dfa, ARGO_FEATURES)
print('ARGO + Lasso RMSE:', round(rmse(df_valid[TARGET], pred_lasso), 1))


## ARGONet, in one note

ARGO uses one location's own search and history. **ARGONet** adds a network step:
borrow a neighbor's recent activity (or its ARGO estimate) as an extra predictor,
choosing neighbors by shared movement or short-lag correlation.

This dataset is a single country (Mexico), so there is no neighbor column to add here.
On a state or county panel you would append a neighbor's lagged signal to `ARGO_FEATURES`
and let the dynamic fit weigh it. That is the real, measurable gain.

**Reflection:** you grew one interpretable regression into ARGO, and every step was an
honest out-of-sample comparison. This is the machinery behind operational flu nowcasting.
